In [ ]:
%%bash
set -euo pipefail

export PATH="/opt/bin:/usr/local/cuda/bin:$PATH"

nvidia-smi -L
nvidia-smi \
    --query-gpu=index,name,compute_cap,driver_version,memory.total \
    --format=csv,noheader


In [ ]:
%%bash
set -euo pipefail

/usr/local/cuda/bin/nvcc --version


In [ ]:
%%bash
set -euo pipefail

apt-get update -qq

DEBIAN_FRONTEND=noninteractive apt-get install -y \
    gcc g++ gfortran \
    openmpi-bin libopenmpi-dev \
    libopenblas-dev liblapack-dev \
    autoconf automake libtool pkg-config \
    cmake make git ca-certificates


In [ ]:
%%bash
set -euo pipefail

NEKO_ROOT=/kaggle/working/neko
NEKO_REPOSITORY="${NEKO_REPOSITORY:-https://github.com/tuananhdao/neko.git}"
NEKO_REF="${NEKO_REF:-euler-gll-idp}"
NEKO_COMMIT="${NEKO_COMMIT:-}"

if [ ! -d "$NEKO_ROOT/.git" ]; then
    git clone --filter=blob:none --no-checkout \
        "$NEKO_REPOSITORY" "$NEKO_ROOT"
fi

if ! git -C "$NEKO_ROOT" diff --quiet ||
   ! git -C "$NEKO_ROOT" diff --cached --quiet; then
    echo "Repository Kaggle có thay đổi tracked; dừng để không ghi đè."
    exit 1
fi

if [ -n "$NEKO_COMMIT" ]; then
    git -C "$NEKO_ROOT" fetch --depth 1 origin "$NEKO_COMMIT"
else
    git -C "$NEKO_ROOT" fetch --depth 1 origin "$NEKO_REF"
fi

RESOLVED_COMMIT="$(git -C "$NEKO_ROOT" rev-parse FETCH_HEAD)"
git -C "$NEKO_ROOT" checkout --detach "$RESOLVED_COMMIT"
git -C "$NEKO_ROOT" rev-parse HEAD | tee /kaggle/working/neko-commit.txt


In [ ]:
%%bash
set -euo pipefail

DEPS_ROOT=/kaggle/working/neko-deps
JSON_ROOT=/kaggle/working/json-fortran
JSON_FORTRAN_REF="${JSON_FORTRAN_REF:-8.3.0}"

if [ -f "$DEPS_ROOT/lib/pkgconfig/json-fortran.pc" ] ||
   [ -f "$DEPS_ROOT/lib64/pkgconfig/json-fortran.pc" ]; then
    echo "JSON-Fortran đã được cài."
else
    if [ ! -d "$JSON_ROOT/.git" ]; then
        git clone --depth 1 --branch "$JSON_FORTRAN_REF" \
            https://github.com/jacobwilliams/json-fortran.git "$JSON_ROOT"
    fi
    cmake -S "$JSON_ROOT" -B "$JSON_ROOT/build" \
        -DCMAKE_BUILD_TYPE=Release \
        -DCMAKE_INSTALL_PREFIX="$DEPS_ROOT" \
        -DUSE_GNU_INSTALL_CONVENTION=ON
    cmake --build "$JSON_ROOT/build" --parallel "$(nproc)"
    cmake --install "$JSON_ROOT/build"
fi


In [ ]:
%%bash
set -euo pipefail

NEKO_INSTALL=/kaggle/working/neko-install

if [ -f "$NEKO_INSTALL/include/neko/neko.mod" ] &&
   [ -x "$NEKO_INSTALL/bin/makeneko" ] &&
   [ -x "$NEKO_INSTALL/bin/neko" ]; then
    echo "Đã có bản cài Neko; notebook sẽ build lại commit đã chọn."
else
    echo "Chưa có bản cài Neko đầy đủ."
fi


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/neko

if [ -f Makefile ]; then
    make clean
fi

bash ./regen.sh


In [ ]:
%%bash
set -euo pipefail

NEKO_ROOT=/kaggle/working/neko
NEKO_INSTALL=/kaggle/working/neko-install
DEPS_ROOT=/kaggle/working/neko-deps

export PATH="/opt/bin:/usr/local/cuda/bin:$PATH"
export PKG_CONFIG_PATH="$DEPS_ROOT/lib/pkgconfig:$DEPS_ROOT/lib64/pkgconfig:${PKG_CONFIG_PATH:-}"
export LD_LIBRARY_PATH="$DEPS_ROOT/lib:$DEPS_ROOT/lib64:/usr/local/cuda/lib64:${LD_LIBRARY_PATH:-}"

COMPUTE_CAP="$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -n 1 | tr -d '.')"
CUDA_ARCH_VALUE="${CUDA_ARCH_OVERRIDE:--arch=sm_${COMPUTE_CAP}}"
echo "CUDA_ARCH=$CUDA_ARCH_VALUE"

cd "$NEKO_ROOT"

FC=gfortran \
CC=gcc \
MPIFC=mpif90 \
MPICC=mpicc \
FCFLAGS="-O3" \
CFLAGS="-O3" \
CUDA_CFLAGS="-O3" \
CUDA_ARCH="$CUDA_ARCH_VALUE" \
NVCC=/usr/local/cuda/bin/nvcc \
./configure \
    --prefix="$NEKO_INSTALL" \
    --enable-real=dp \
    --with-cuda=/usr/local/cuda


In [ ]:
%%bash
set -euo pipefail

NEKO_ROOT=/kaggle/working/neko

grep "NEKO_BCKND_CUDA" "$NEKO_ROOT/src/config/neko_config.f90"
echo "Commit được build: $(git -C "$NEKO_ROOT" rev-parse HEAD)"


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/neko
make -j"$(nproc)"


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/neko
make install


In [ ]:
%%bash
set -euo pipefail

NEKO_INSTALL=/kaggle/working/neko-install

ls -l \
    "$NEKO_INSTALL/bin/makeneko" \
    "$NEKO_INSTALL/bin/neko" \
    "$NEKO_INSTALL/include/neko/neko.mod"

find "$NEKO_INSTALL/lib" -maxdepth 1 -name 'libneko*' -print


In [ ]:
%%bash
set -euo pipefail

NEKO_INSTALL=/kaggle/working/neko-install

export PATH="$NEKO_INSTALL/bin:$PATH"

python - <<'PY'
import numpy
import pytest
print(f"numpy={numpy.__version__}")
print(f"pytest={pytest.__version__}")
PY

GPU_COUNT="$(nvidia-smi -L | wc -l)"
test "$GPU_COUNT" -ge 1
echo "Số GPU khả dụng: $GPU_COUNT"
test -x "$NEKO_INSTALL/bin/makeneko"


In [ ]:
%%bash
set -euo pipefail

NEKO_INSTALL=/kaggle/working/neko-install
DEPS_ROOT=/kaggle/working/neko-deps
ARTIFACT_DIR=/kaggle/working/artifacts
TEST_LOG="$ARTIFACT_DIR/euler_idp_cuda_pytest.log"
JUNIT_FILE="$ARTIFACT_DIR/euler_idp_cuda_junit.xml"

export PATH="$NEKO_INSTALL/bin:$PATH"
export LD_LIBRARY_PATH="/usr/local/cuda/lib64:$NEKO_INSTALL/lib:$DEPS_ROOT/lib:$DEPS_ROOT/lib64:${LD_LIBRARY_PATH:-}"
export OMPI_ALLOW_RUN_AS_ROOT=1
export OMPI_ALLOW_RUN_AS_ROOT_CONFIRM=1

mkdir -p "$ARTIFACT_DIR"
GPU_COUNT="$(nvidia-smi -L | wc -l)"
if [ "$GPU_COUNT" -ge 2 ]; then
    MAX_NPROCS=2
else
    MAX_NPROCS=1
fi
echo "Chạy kiểm thử CUDA với tối đa $MAX_NPROCS MPI rank."
cd /kaggle/working/neko/tests/integration
if [ -d logs ]; then
    find logs -maxdepth 1 -type f -name '*.log' -delete
fi

set +e
python -m pytest -vv tests/test_euler_idp \
    --backend=cuda \
    --launcher-script=./default_cuda_launcher.sh \
    --max_nprocs="$MAX_NPROCS" \
    --real_precision=dp \
    --junitxml="$JUNIT_FILE" 2>&1 | tee "$TEST_LOG"
RUN_STATUS=${PIPESTATUS[0]}
set -e

CUDA_CASE_LOGS="$ARTIFACT_DIR/euler_idp_cuda_case_logs"
rm -rf "$CUDA_CASE_LOGS"
if [ -d logs ]; then
    cp -R logs "$CUDA_CASE_LOGS"
    tar -czf "$ARTIFACT_DIR/euler_idp_cuda_case_logs.tar.gz" \
        -C "$CUDA_CASE_LOGS" .
fi
if [ "$RUN_STATUS" -ne 0 ]; then
    echo "Bộ kiểm thử Euler IDP CUDA thất bại với mã $RUN_STATUS"
    exit "$RUN_STATUS"
fi
echo "Bộ kiểm thử Euler IDP CUDA hoàn tất; full log: $TEST_LOG"


In [ ]:
%%bash
set -euo pipefail

NEKO_ROOT=/kaggle/working/neko
CPU_ROOT=/kaggle/working/neko-cpu
CPU_INSTALL=/kaggle/working/neko-cpu-install
DEPS_ROOT=/kaggle/working/neko-deps
RESOLVED_COMMIT="$(cat /kaggle/working/neko-commit.txt)"

if [ ! -d "$CPU_ROOT/.git" ]; then
    git clone --shared --no-checkout "$NEKO_ROOT" "$CPU_ROOT"
fi
if ! git -C "$CPU_ROOT" diff --quiet ||
   ! git -C "$CPU_ROOT" diff --cached --quiet; then
    echo "Repository CPU có thay đổi tracked; dừng để không ghi đè."
    exit 1
fi
git -C "$CPU_ROOT" fetch --depth 1 "$NEKO_ROOT" "$RESOLVED_COMMIT"
git -C "$CPU_ROOT" checkout --detach FETCH_HEAD

export PKG_CONFIG_PATH="$DEPS_ROOT/lib/pkgconfig:$DEPS_ROOT/lib64/pkgconfig:${PKG_CONFIG_PATH:-}"
export LD_LIBRARY_PATH="$DEPS_ROOT/lib:$DEPS_ROOT/lib64:${LD_LIBRARY_PATH:-}"

cd "$CPU_ROOT"
if [ -f Makefile ]; then
    make clean
fi
bash ./regen.sh
FC=gfortran \
CC=gcc \
MPIFC=mpif90 \
MPICC=mpicc \
FCFLAGS="-O3" \
CFLAGS="-O3" \
./configure \
    --prefix="$CPU_INSTALL" \
    --enable-real=dp
make -j"$(nproc)"
make install
echo "CPU reference commit: $(git rev-parse HEAD)"


In [ ]:
%%bash
set -euo pipefail

CPU_ROOT=/kaggle/working/neko-cpu
CPU_INSTALL=/kaggle/working/neko-cpu-install
DEPS_ROOT=/kaggle/working/neko-deps
ARTIFACT_DIR=/kaggle/working/artifacts
TEST_LOG="$ARTIFACT_DIR/euler_idp_cpu_pytest.log"
JUNIT_FILE="$ARTIFACT_DIR/euler_idp_cpu_junit.xml"
CPU_CASE_LOGS="$ARTIFACT_DIR/euler_idp_cpu_case_logs"

export PATH="$CPU_INSTALL/bin:$PATH"
export LD_LIBRARY_PATH="$CPU_INSTALL/lib:$DEPS_ROOT/lib:$DEPS_ROOT/lib64:${LD_LIBRARY_PATH:-}"
export OMPI_ALLOW_RUN_AS_ROOT=1
export OMPI_ALLOW_RUN_AS_ROOT_CONFIRM=1

mkdir -p "$ARTIFACT_DIR"
cd "$CPU_ROOT/tests/integration"
if [ -d logs ]; then
    find logs -maxdepth 1 -type f -name '*.log' -delete
fi

set +e
python -m pytest -vv tests/test_euler_idp \
    --backend=cpu \
    --launcher-script=./default_cpu_launcher.sh \
    --max_nprocs=2 \
    --real_precision=dp \
    --junitxml="$JUNIT_FILE" 2>&1 | tee "$TEST_LOG"
RUN_STATUS=${PIPESTATUS[0]}
set -e

rm -rf "$CPU_CASE_LOGS"
if [ -d logs ]; then
    cp -R logs "$CPU_CASE_LOGS"
    tar -czf "$ARTIFACT_DIR/euler_idp_cpu_case_logs.tar.gz" \
        -C "$CPU_CASE_LOGS" .
fi
if [ "$RUN_STATUS" -ne 0 ]; then
    echo "Bộ kiểm thử Euler IDP CPU thất bại với mã $RUN_STATUS"
    exit "$RUN_STATUS"
fi
echo "Bộ kiểm thử CPU reference hoàn tất; full log: $TEST_LOG"


In [ ]:
%%bash
set -euo pipefail

NEKO_ROOT=/kaggle/working/neko
ARTIFACT_DIR=/kaggle/working/artifacts

python "$NEKO_ROOT/tests/integration/tests/test_euler_idp/compare_backend_summaries.py" \
    --cpu-logs "$ARTIFACT_DIR/euler_idp_cpu_case_logs" \
    --device-logs "$ARTIFACT_DIR/euler_idp_cuda_case_logs" \
    --precision dp \
    --output "$ARTIFACT_DIR/euler_idp_backend_parity.json"


In [ ]:
%%bash
set -euo pipefail

NEKO_ROOT=/kaggle/working/neko
ARTIFACT_DIR=/kaggle/working/artifacts
METADATA_FILE="$ARTIFACT_DIR/environment.txt"

mkdir -p "$ARTIFACT_DIR"
cp "$NEKO_ROOT/config.log" "$ARTIFACT_DIR/config.log"

{
    echo "commit=$(git -C "$NEKO_ROOT" rev-parse HEAD)"
    echo "branch=$(git -C "$NEKO_ROOT" branch --show-current)"
    nvidia-smi \
        --query-gpu=index,name,compute_cap,driver_version,memory.total \
        --format=csv,noheader
    /usr/local/cuda/bin/nvcc --version
    python -m pytest --version
} >"$METADATA_FILE"

find "$ARTIFACT_DIR" -maxdepth 1 -type f \
    ! -name checksums.sha256 -print0 | sort -z | \
    xargs -0 sha256sum >"$ARTIFACT_DIR/checksums.sha256"
cat "$ARTIFACT_DIR/checksums.sha256"
find "$ARTIFACT_DIR" -maxdepth 1 -type f -printf '%f\n' | sort
